# Guwahati cloud and visibility LSTM

Run the cells in order. This notebook uses the bounded cloud-fill dataset, excludes imputed values from targets, uses chronological splits, and saves models and metrics under `10_models/cloud_visibility_lstm`.

In [1]:
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PROJECT_ROOT = Path.cwd().parent
DATA_FILE = PROJECT_ROOT / "08_unified_dataset" / "guwahati_unified_lstm_dataset_v2_bounded_cloud_fill_20240811_20260131.csv"
MODEL_DIR = PROJECT_ROOT / "10_models" / "cloud_visibility_lstm"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

HISTORY_STEPS = 60
HORIZON_STEPS = 15
TRAIN_END = pd.Timestamp("2025-08-31 23:59:59")
VALIDATION_END = pd.Timestamp("2025-10-31 23:59:59")
BATCH_SIZE = 256
EPOCHS = 40

TARGETS = [
    "CLOUD_Cloud_Base_1", "CLOUD_Cloud_Base_2", "CLOUD_Cloud_Base_3",
    "CLOUD_Vertical_Visibility", "VIS_Visibility_MOR",
]
IMPUTED_FLAGS = {
    "CLOUD_Cloud_Base_1": "CLOUD_Cloud_Base_1_IMPUTED",
    "CLOUD_Cloud_Base_2": "CLOUD_Cloud_Base_2_IMPUTED",
    "CLOUD_Cloud_Base_3": "CLOUD_Cloud_Base_3_IMPUTED",
    "CLOUD_Vertical_Visibility": "CLOUD_Vertical_Visibility_IMPUTED",
}
FEATURES = [
    "CLOUD_Cloud_Base_1", "CLOUD_Cloud_Base_2", "CLOUD_Cloud_Base_3",
    "CLOUD_OCTA1", "CLOUD_OCTA2", "CLOUD_OCTA3", "CLOUD_Vertical_Visibility",
    "PTU_Pressure_QNH", "PTU_Temperature", "PTU_DewPoint", "PTU_Humidity",
    "PW_Rain_Intensity", "PW_Rain_Sum", "VIS_Visibility_MOR", "VIS_RVR",
    "WIND_Wind_Speed", "WIND_Wind_Gust", "WIND_Wind_Dir_Sin", "WIND_Wind_Dir_Cos",
    "Hour_Sin", "Hour_Cos", "CLOUD_SENSOR_ON", "PTU_SENSOR_ON", "PW_SENSOR_ON",
    "VIS_SENSOR_ON", "WIND_SENSOR_ON", "CLOUD_Cloud_Base_1_IMPUTED",
    "CLOUD_Cloud_Base_2_IMPUTED", "CLOUD_Cloud_Base_3_IMPUTED",
    "CLOUD_Vertical_Visibility_IMPUTED", "CLOUD_LONG_GAP", "CLOUD_VERTICAL_LONG_GAP",
]
assert DATA_FILE.exists(), f"Dataset not found: {DATA_FILE}"
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)


PyTorch: 2.5.1
Device: cuda


In [2]:
raw = pd.read_csv(DATA_FILE, parse_dates=["Datetime"], dayfirst=True)
raw = raw.sort_values("Datetime").drop_duplicates("Datetime").set_index("Datetime")
engineered_features = {"WIND_Wind_Dir_Sin", "WIND_Wind_Dir_Cos"}
raw_required = sorted((set(FEATURES + TARGETS) - engineered_features) - set(raw.columns))
assert not raw_required, f"Missing raw columns: {raw_required}"
numeric_columns = sorted((set(FEATURES + TARGETS) - engineered_features) & set(raw.columns))
raw[numeric_columns] = raw[numeric_columns].apply(pd.to_numeric, errors="coerce")
raw.loc[raw["WIND_Wind_Speed"] < 0, "WIND_Wind_Speed"] = np.nan
raw.loc[raw["WIND_Wind_Gust"] < 0, "WIND_Wind_Gust"] = np.nan

minute = raw.resample("1min").last()
minute["WIND_Wind_Dir_Sin"] = np.sin(np.deg2rad(minute["WIND_Wind_Dir"]))
minute["WIND_Wind_Dir_Cos"] = np.cos(np.deg2rad(minute["WIND_Wind_Dir"]))
minute = minute.drop(columns=["WIND_Wind_Dir"], errors="ignore").reset_index()
minute["ROW_GAP"] = minute["Datetime"].diff().dt.total_seconds().gt(60).fillna(False)
print("Rows:", len(minute), "range:", minute["Datetime"].min(), minute["Datetime"].max())
for target in TARGETS:
    valid = minute[target].notna()
    if target in IMPUTED_FLAGS:
        valid &= minute[IMPUTED_FLAGS[target]].fillna(0).eq(0)
    print(target, "usable labels:", int(valid.sum()))


Rows: 775329 range: 2024-08-11 00:00:00 2026-01-31 10:08:00
CLOUD_Cloud_Base_1 usable labels: 235834
CLOUD_Cloud_Base_2 usable labels: 107602
CLOUD_Cloud_Base_3 usable labels: 37925
CLOUD_Vertical_Visibility usable labels: 6489
VIS_Visibility_MOR usable labels: 472929


In [3]:
feature_frame = minute.set_index("Datetime")[FEATURES]
train_rows = feature_frame.index <= TRAIN_END
scaler = StandardScaler()
train_medians = feature_frame.loc[train_rows].median()
scaler.fit(feature_frame.loc[train_rows].fillna(train_medians))
prepared = pd.DataFrame(
    scaler.transform(feature_frame.fillna(train_medians)),
    index=feature_frame.index, columns=FEATURES,
).replace([np.inf, -np.inf], np.nan).fillna(0).astype("float32")
with open(MODEL_DIR / "feature_scaler.json", "w", encoding="utf-8") as handle:
    json.dump({"features": FEATURES, "medians": train_medians.to_dict()}, handle, indent=2)

def target_is_observed(target):
    observed = minute[target].notna()
    flag = IMPUTED_FLAGS.get(target)
    if flag:
        observed &= minute[flag].fillna(0).eq(0)
    return observed.to_numpy()

def make_windows(target):
    observed = target_is_observed(target)
    values = prepared.to_numpy(dtype="float32")
    target_values = minute[target].to_numpy(dtype="float32")
    gaps = minute["ROW_GAP"].to_numpy(dtype=bool)
    times = minute["Datetime"].to_numpy()
    xs, ys, baselines, timestamps = [], [], [], []
    for end in range(HISTORY_STEPS, len(minute) - HORIZON_STEPS + 1):
        horizon = end + HORIZON_STEPS - 1
        if not observed[horizon] or not observed[end - 1] or gaps[end:horizon + 1].any():
            continue
        xs.append(values[end - HISTORY_STEPS:end])
        ys.append(target_values[horizon])
        baselines.append(target_values[end - 1])
        timestamps.append(times[horizon])
    return np.asarray(xs), np.asarray(ys), np.asarray(baselines), pd.to_datetime(timestamps)


In [4]:
class CloudVisibilityLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(len(FEATURES), 64, num_layers=2, batch_first=True, dropout=0.2)
        self.head = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1))

    def forward(self, inputs):
        output, _ = self.lstm(inputs)
        return self.head(output[:, -1, :]).squeeze(-1)

def time_split(x, y, baseline, timestamps):
    train = timestamps <= TRAIN_END
    validation = (timestamps > TRAIN_END) & (timestamps <= VALIDATION_END)
    test = timestamps > VALIDATION_END
    return (x[train], y[train], baseline[train]), (x[validation], y[validation], baseline[validation]), (x[test], y[test], baseline[test])

results = []
for target in TARGETS:
    x, y, baseline, timestamps = make_windows(target)
    (x_train, y_train, _), (x_val, y_val, _), (x_test, y_test, baseline_test) = time_split(x, y, baseline, timestamps)
    print(target, "windows:", len(x_train), len(x_val), len(x_test))
    if min(len(x_train), len(x_val), len(x_test)) == 0:
        continue
    model = CloudVisibilityLSTM().to(DEVICE)
    target_mean = float(np.mean(y_train))
    target_std = float(np.std(y_train))
    if target_std == 0:
        target_std = 1.0
    y_train_scaled = ((y_train - target_mean) / target_std).astype("float32")
    y_val_scaled = ((y_val - target_mean) / target_std).astype("float32")
    optimizer = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-5)
    loss_function = nn.HuberLoss(delta=1.0)
    train_loader = DataLoader(TensorDataset(torch.from_numpy(x_train), torch.from_numpy(y_train_scaled)), batch_size=BATCH_SIZE, shuffle=False)
    validation_loader = DataLoader(TensorDataset(torch.from_numpy(x_val), torch.from_numpy(y_val_scaled)), batch_size=BATCH_SIZE, shuffle=False)
    best_validation_loss = float("inf")
    best_state = None
    patience_counter = 0
    for epoch in range(EPOCHS):
        model.train()
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss_function(model(inputs), labels).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        model.eval()
        validation_losses = []
        with torch.no_grad():
            for inputs, labels in validation_loader:
                validation_losses.append(loss_function(model(inputs.to(DEVICE)), labels.to(DEVICE)).item())
        validation_loss = float(np.mean(validation_losses))
        print(f"{target} epoch {epoch + 1}/{EPOCHS}, val_loss={validation_loss:.5f}")
        if validation_loss < best_validation_loss:
            best_validation_loss = validation_loss
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= 6:
                break
    model.load_state_dict(best_state)
    torch.save({"model_state_dict": model.state_dict(), "features": FEATURES, "target_mean": target_mean, "target_std": target_std}, MODEL_DIR / f"{target}.pt")
    model.eval()
    test_loader = DataLoader(TensorDataset(torch.from_numpy(x_test)), batch_size=BATCH_SIZE, shuffle=False)
    predictions = []
    with torch.no_grad():
        for (inputs,) in test_loader:
            predictions.append(model(inputs.to(DEVICE)).cpu().numpy())
    prediction = np.concatenate(predictions) * target_std + target_mean
    results.append({
        "target": target, "train_windows": len(x_train), "validation_windows": len(x_val), "test_windows": len(x_test),
        "lstm_mae": mean_absolute_error(y_test, prediction), "lstm_rmse": np.sqrt(mean_squared_error(y_test, prediction)),
        "lstm_r2": r2_score(y_test, prediction),
        "persistence_mae": mean_absolute_error(y_test, baseline_test), "persistence_rmse": np.sqrt(mean_squared_error(y_test, baseline_test)),
        "persistence_r2": r2_score(y_test, baseline_test),
    })
metrics = pd.DataFrame(results)
metrics.to_csv(MODEL_DIR / "metrics.csv", index=False)
display(metrics)


CLOUD_Cloud_Base_1 windows: 180048 11391 28106
CLOUD_Cloud_Base_1 epoch 1/40, val_loss=0.10346
CLOUD_Cloud_Base_1 epoch 2/40, val_loss=0.09309
CLOUD_Cloud_Base_1 epoch 3/40, val_loss=0.08997
CLOUD_Cloud_Base_1 epoch 4/40, val_loss=0.08757
CLOUD_Cloud_Base_1 epoch 5/40, val_loss=0.08651
CLOUD_Cloud_Base_1 epoch 6/40, val_loss=0.08545
CLOUD_Cloud_Base_1 epoch 7/40, val_loss=0.08550
CLOUD_Cloud_Base_1 epoch 8/40, val_loss=0.08513
CLOUD_Cloud_Base_1 epoch 9/40, val_loss=0.08553
CLOUD_Cloud_Base_1 epoch 10/40, val_loss=0.08725
CLOUD_Cloud_Base_1 epoch 11/40, val_loss=0.08774
CLOUD_Cloud_Base_1 epoch 12/40, val_loss=0.08980
CLOUD_Cloud_Base_1 epoch 13/40, val_loss=0.09160
CLOUD_Cloud_Base_1 epoch 14/40, val_loss=0.09679
CLOUD_Cloud_Base_2 windows: 75269 4594 6109
CLOUD_Cloud_Base_2 epoch 1/40, val_loss=0.17915
CLOUD_Cloud_Base_2 epoch 2/40, val_loss=0.17336
CLOUD_Cloud_Base_2 epoch 3/40, val_loss=0.17162
CLOUD_Cloud_Base_2 epoch 4/40, val_loss=0.17086
CLOUD_Cloud_Base_2 epoch 5/40, val_loss=

,target,train_windows,validation_windows,test_windows,lstm_mae,lstm_rmse,lstm_r2,persistence_mae,persistence_rmse,persistence_r2
0,CLOUD_Cloud_Base_1,180048,11391,28106,1031.719971,2215.815990,0.826800,644.177429,2294.290522,0.814315
1,CLOUD_Cloud_Base_2,75269,4594,6109,1782.508423,3068.556338,0.747042,1374.251099,3253.510566,0.715629
2,CLOUD_Cloud_Base_3,18853,1036,858,1877.990845,2925.304770,0.819137,1313.053589,2827.186499,0.831066
3,CLOUD_Vertical_Visibility,2609,480,1006,96.420319,268.830874,0.718208,27.053679,107.829627,0.954664
4,VIS_Visibility_MOR,335598,24451,100279,588.082092,869.872153,0.914126,507.706970,891.171666,0.909869


In [5]:
metrics = pd.read_csv(MODEL_DIR / "metrics.csv")
metrics["mae_improvement_percent"] = 100 * (metrics["persistence_mae"] - metrics["lstm_mae"]) / metrics["persistence_mae"]
metrics["r2_difference"] = metrics["lstm_r2"] - metrics["persistence_r2"]
display(metrics.sort_values("mae_improvement_percent", ascending=False))


,target,train_windows,validation_windows,test_windows,lstm_mae,lstm_rmse,lstm_r2,persistence_mae,persistence_rmse,persistence_r2,mae_improvement_percent,r2_difference
4,VIS_Visibility_MOR,335598,24451,100279,588.082092,869.872153,0.914126,507.706970,891.171666,0.909869,-15.831006,0.004257
1,CLOUD_Cloud_Base_2,75269,4594,6109,1782.508423,3068.556338,0.747042,1374.251099,3253.510566,0.715629,-29.707622,0.031413
2,CLOUD_Cloud_Base_3,18853,1036,858,1877.990845,2925.304770,0.819137,1313.053589,2827.186499,0.831066,-43.024692,-0.011929
0,CLOUD_Cloud_Base_1,180048,11391,28106,1031.719971,2215.815990,0.826800,644.177429,2294.290522,0.814315,-60.160838,0.012485
3,CLOUD_Vertical_Visibility,2609,480,1006,96.420319,268.830874,0.718208,27.053679,107.829627,0.954664,-256.403727,-0.236456
